# Fantasy Football Lineup Optimizer — Quantum QUBO Solver

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/qumulator/qumulator-sdk/blob/main/notebooks/fantasy_football.ipynb)
[![Qumulator](https://img.shields.io/badge/powered%20by-Qumulator-7c6fff.svg)](https://qumulator.com)

**What this notebook does:** Selects the highest-scoring DraftKings daily fantasy
football lineup from a player pool, subject to salary cap and roster slot constraints —
using Qumulator's quantum ground-state solver.

**No quantum physics knowledge required.** Edit the player pool and projected points
below, then press *Run All*.

### The problem

DraftKings daily fantasy: pick **9 players** (QB + 2 RB + 3 WR + TE + FLEX + DST)
from a pool of ~40 candidates. Total salary must be under **$50,000**.
Maximise total projected fantasy points.

With 40 players, there are over **300 million** possible 9-player combinations.
Classical greedy heuristics miss globally optimal solutions by 3-10 points —
which is often the difference between winning and losing a contest.

### Why quantum?

This is a **Quadratic Unconstrained Binary Optimisation (QUBO)** problem — the native
language of quantum annealing and ground-state solvers. Qumulator's KLT engine finds the
minimum-energy configuration of the equivalent Ising spin Hamiltonian, corresponding to
the maximum-score valid lineup.


In [ ]:
# Set your API key ---------------------------------------------------------------
# Free key (no credit card): https://qumulator.com
import os
API_KEY = os.environ.get("QUMULATOR_API_KEY", "YOUR_KEY_HERE")
API_URL = "https://api.qumulator.com"


In [ ]:
%pip install qumulator-sdk --quiet
from qumulator import QumulatorClient
import numpy as np
import time

client = QumulatorClient(api_url=API_URL, api_key=API_KEY)
print("SDK ready.")


## Step 1 — Player pool

Edit the `proj` column with this week's projected points from FantasyPros, DraftKings,
or your own model. Everything else (names, salaries) is also editable.

Each entry: `(position, name, salary, projected_points)`


In [ ]:
# Player pool ----------------------------------------------------------------
# Format: (position, name, DK_salary, projected_points)
# Positions: QB, RB, WR, TE, DST

PLAYER_POOL = [
    # QBs
    ("QB", "Lamar Jackson",       8200, 32.5),
    ("QB", "Josh Allen",          8000, 31.8),
    ("QB", "Jalen Hurts",         7800, 30.4),
    ("QB", "Patrick Mahomes",     7600, 29.9),
    ("QB", "Tua Tagovailoa",      6800, 26.1),
    ("QB", "Sam Darnold",         5500, 20.3),

    # RBs
    ("RB", "Christian McCaffrey", 9200, 28.4),
    ("RB", "Derrick Henry",       7800, 25.1),
    ("RB", "Saquon Barkley",      7400, 23.5),
    ("RB", "Breece Hall",         7200, 22.8),
    ("RB", "De'Von Achane",       7000, 21.9),
    ("RB", "Josh Jacobs",         6800, 20.7),
    ("RB", "Joe Mixon",           6400, 18.9),
    ("RB", "Aaron Jones",         5800, 16.4),
    ("RB", "Tony Pollard",        5600, 15.8),
    ("RB", "Zack Moss",           4800, 13.1),

    # WRs
    ("WR", "Tyreek Hill",         8600, 27.3),
    ("WR", "CeeDee Lamb",         8400, 26.8),
    ("WR", "Ja'Marr Chase",       8200, 26.1),
    ("WR", "A.J. Brown",          7400, 23.1),
    ("WR", "Stefon Diggs",        7200, 22.4),
    ("WR", "Davante Adams",       7000, 21.9),
    ("WR", "Amon-Ra St. Brown",   6800, 20.5),
    ("WR", "Brandon Aiyuk",       6600, 20.1),
    ("WR", "Rashee Rice",         6400, 19.4),
    ("WR", "Calvin Ridley",       6200, 18.9),
    ("WR", "DeAndre Hopkins",     5800, 17.2),
    ("WR", "Puka Nacua",          5600, 16.1),

    # TEs
    ("TE", "Travis Kelce",        8000, 22.3),
    ("TE", "Mark Andrews",        6600, 18.4),
    ("TE", "Trey McBride",        5800, 16.1),
    ("TE", "Dallas Goedert",      5600, 15.4),
    ("TE", "Sam LaPorta",         5400, 15.8),
    ("TE", "Jake Ferguson",       4800, 12.7),

    # DST
    ("DST", "San Francisco 49ers", 4200, 11.5),
    ("DST", "Baltimore Ravens",    4000, 10.8),
    ("DST", "Dallas Cowboys",      3800, 10.1),
    ("DST", "Buffalo Bills",       3600,  9.8),
    ("DST", "New England Patriots",3000,  7.4),
    ("DST", "New York Jets",       2800,  6.9),
]

# DraftKings roster format
SALARY_CAP     = 50_000
ROSTER         = {"QB": 1, "RB": 2, "WR": 3, "TE": 1, "FLEX": 1, "DST": 1}
FLEX_POSITIONS = {"RB", "WR", "TE"}

N = len(PLAYER_POOL)
print(f"Player pool : {N} players")
print(f"Salary cap  : ${SALARY_CAP:,}")
print(f"Roster      : {sum(ROSTER.values())} players total  (QB + 2RB + 3WR + TE + FLEX + DST)")


## Step 2 — Build the QUBO

The optimisation problem is:

    maximise   sum_i  p_i * x_i          (projected points)
    subject to sum_i  s_i * x_i <= B     (salary cap)
               position count constraints
               x_i in {0, 1}

We encode the constraints as **quadratic penalty terms** added to the QUBO objective.
The KLT solver then finds the binary vector x* that minimises this combined objective.


In [ ]:
# QUBO penalty weights -------------------------------------------------------
# Higher = constraint enforced more tightly (objective signal gets weaker)
LAMBDA_POS  = 15.0    # per-player over/under-count penalty
LAMBDA_FLEX = 10.0    # FLEX slot penalty
LAMBDA_CAP  = 0.003   # salary cap penalty (per dollar^2)


def build_qubo(players, salary_cap, roster, flex_positions,
               lam_pos=LAMBDA_POS, lam_flex=LAMBDA_FLEX, lam_cap=LAMBDA_CAP):
    n = len(players)
    Q = np.zeros((n, n))

    # Objective: maximise projected points (minimise negative points)
    for i, (pos, name, sal, proj) in enumerate(players):
        Q[i, i] -= proj

    # Constraint 1: position counts (excluding FLEX)
    for pos, count in roster.items():
        if pos == "FLEX":
            continue
        idx = [i for i, p in enumerate(players) if p[0] == pos]
        # penalty: lam * (sum x_i - count)^2
        for i in idx:
            Q[i, i] += lam_pos * (1 - 2 * count)
        for a, i in enumerate(idx):
            for j in idx[a + 1:]:
                Q[i, j] += 2 * lam_pos
                Q[j, i] += 2 * lam_pos

    # Constraint 2: FLEX slot (one extra eligible player above base counts)
    flex_target = sum(roster.get(p, 0) for p in flex_positions) + 1
    fidx = [i for i, p in enumerate(players) if p[0] in flex_positions]
    for i in fidx:
        Q[i, i] += lam_flex * (1 - 2 * flex_target)
    for a, i in enumerate(fidx):
        for j in fidx[a + 1:]:
            Q[i, j] += 2 * lam_flex
            Q[j, i] += 2 * lam_flex

    # Constraint 3: salary cap penalty  lam * (sum s_i x_i - B)^2
    sals = np.array([p[2] for p in players], dtype=float)
    for i in range(n):
        Q[i, i] += lam_cap * sals[i] * (sals[i] - 2 * salary_cap)
    for i in range(n):
        for j in range(i + 1, n):
            Q[i, j] += 2 * lam_cap * sals[i] * sals[j]
            Q[j, i] += 2 * lam_cap * sals[i] * sals[j]

    return Q


Q = build_qubo(PLAYER_POOL, SALARY_CAP, ROSTER, FLEX_POSITIONS)
print(f"QUBO matrix : {Q.shape}  ({np.count_nonzero(Q)} non-zero entries)")
print(f"Value range : [{Q.min():.2f}, {Q.max():.2f}]")


## Step 3 — Solve with Qumulator's KLT engine

The QUBO is mapped to an Ising spin Hamiltonian via the substitution
x_i = (1 - sigma_i) / 2, sigma_i in {-1, +1}:

    H = sum_{i<j}  J_{ij} sigma_i sigma_j  +  sum_i  h_i sigma_i

where J_{ij} = Q_{ij} / 4 for i != j.
Qumulator's KLT engine finds the ground state — the spin configuration minimising H.


In [ ]:
# Convert QUBO to Ising interaction matrix
# KLT minimises  E = -1/2 * s^T J s,  so we pass J = -Q/4
J_ising = -Q / 4.0

print(f"Submitting {Q.shape[0]}-variable Ising problem to Qumulator KLT engine...")
t0 = time.perf_counter()
klt_result = client.klt.run(
    interaction_matrix=J_ising.tolist(),
    confinement_strength=0.05,
    cluster_size=2,
)
elapsed = time.perf_counter() - t0
print(f"KLT solver completed in {elapsed:.2f}s")
print(f"KLT ground-state energy: {klt_result.energy:.4f}")


In [ ]:
# Extract binary selections from KLT phases ----------------------------------
# KLT returns continuous phases phi_i in [0, 2*pi).
# cos(phi_i) > 0  ->  spin up  ->  player selected (x_i = 1)
phases = np.array(klt_result.states)
x_klt  = (np.cos(phases) > 0).astype(int)

print(f"KLT raw selection: {x_klt.sum()} players selected")
print(f"  Salary : ${sum(PLAYER_POOL[i][2] for i in range(N) if x_klt[i]):,}")
print(f"  Points : {sum(PLAYER_POOL[i][3] for i in range(N) if x_klt[i]):.1f}")


## Step 4 — Repair into a valid DraftKings lineup

The KLT output is a continuous relaxation. We use the phase values as a
**ranking signal** and greedily fill each roster slot from the highest-ranked
eligible players, respecting salary cap and position limits.


In [ ]:
def repair_lineup(phases, players, salary_cap, roster, flex_positions):
    klt_rank = np.cos(phases)    # higher = more "selected"
    order    = np.argsort(-klt_rank)

    # Fill cheapest positions first to preserve cap room for star players
    slot_specs = (
        [("DST", "DST")] * roster["DST"] +
        [("TE",  "TE")]  * roster["TE"]  +
        [("QB",  "QB")]  * roster["QB"]  +
        [("RB",  "RB")]  * roster["RB"]  +
        [("WR",  "WR")]  * roster["WR"]  +
        [("FLEX","FLEX")] * roster["FLEX"]
    )

    def min_remaining_cost(start_k, used_set):
        cost      = 0
        temp_used = set(used_set)
        for _, pf in slot_specs[start_k:]:
            best_sal = None; best_i = None
            for i, (pos, name, sal, proj) in enumerate(players):
                if i in temp_used: continue
                ok = (pf == "FLEX" and pos in flex_positions) or pos == pf
                if ok and (best_sal is None or sal < best_sal):
                    best_sal = sal; best_i = i
            if best_i is not None:
                cost += best_sal; temp_used.add(best_i)
        return cost

    lineup = []; used = set(); budget = salary_cap
    for k, (slot, pf) in enumerate(slot_specs):
        future_need = min_remaining_cost(k + 1, used)
        best_idx = None; best_rank = -99.0
        for i in order:
            if i in used: continue
            pos, name, sal, proj = players[i]
            ok = (pf == "FLEX" and pos in flex_positions) or pos == pf
            if not ok or sal + future_need > budget: continue
            if klt_rank[i] > best_rank:
                best_rank = klt_rank[i]; best_idx = i
        if best_idx is None:
            for i in order[::-1]:
                if i in used: continue
                pos, name, sal, proj = players[i]
                ok = (pf == "FLEX" and pos in flex_positions) or pos == pf
                if ok and sal <= budget: best_idx = i; break
        if best_idx is not None:
            pos, name, sal, proj = players[best_idx]
            lineup.append({"idx": best_idx, "pos": pos, "slot": slot,
                           "name": name, "sal": sal, "proj": proj})
            used.add(best_idx); budget -= sal

    return lineup


lineup = repair_lineup(phases, PLAYER_POOL, SALARY_CAP, ROSTER, FLEX_POSITIONS)
total_pts = sum(p["proj"] for p in lineup)
total_sal = sum(p["sal"]  for p in lineup)
print(f"Repaired lineup: {len(lineup)} players  |  ${total_sal:,} salary  |  {total_pts:.1f} pts")


## Step 5 — Classical greedy baseline


In [ ]:
def greedy_lineup(players, salary_cap, roster, flex_positions):
    # Budget-horizon-aware greedy: ranks by projected pts / salary efficiency.
    eff_rank = np.array([p[3] / p[2] for p in players])
    order    = np.argsort(-eff_rank)

    slot_specs = (
        [("DST", "DST")] * roster["DST"] +
        [("TE",  "TE")]  * roster["TE"]  +
        [("QB",  "QB")]  * roster["QB"]  +
        [("RB",  "RB")]  * roster["RB"]  +
        [("WR",  "WR")]  * roster["WR"]  +
        [("FLEX","FLEX")] * roster["FLEX"]
    )

    def min_remaining_cost(start_k, used_set):
        cost      = 0
        temp_used = set(used_set)
        for _, pf in slot_specs[start_k:]:
            best_sal = None; best_i = None
            for i, (pos, name, sal, proj) in enumerate(players):
                if i in temp_used: continue
                ok = (pf == "FLEX" and pos in flex_positions) or pos == pf
                if ok and (best_sal is None or sal < best_sal):
                    best_sal = sal; best_i = i
            if best_i is not None:
                cost += best_sal; temp_used.add(best_i)
        return cost

    lineup = []; used = set(); budget = salary_cap

    for k, (slot, pf) in enumerate(slot_specs):
        future_need = min_remaining_cost(k + 1, used)
        best_idx = None; best_eff = -99.0
        for i in order:
            if i in used: continue
            pos, name, sal, proj = players[i]
            ok = (pf == "FLEX" and pos in flex_positions) or pos == pf
            if not ok or sal + future_need > budget: continue
            if eff_rank[i] > best_eff:
                best_eff = eff_rank[i]; best_idx = i
        if best_idx is None:
            for i in order[::-1]:
                if i in used: continue
                pos, name, sal, proj = players[i]
                ok = (pf == "FLEX" and pos in flex_positions) or pos == pf
                if ok and sal <= budget: best_idx = i; break
        if best_idx is not None:
            pos, name, sal, proj = players[best_idx]
            lineup.append({"idx": best_idx, "pos": pos, "slot": slot,
                           "name": name, "sal": sal, "proj": proj})
            used.add(best_idx); budget -= sal
    return lineup


greedy = greedy_lineup(PLAYER_POOL, SALARY_CAP, ROSTER, FLEX_POSITIONS)
g_pts  = sum(p["proj"] for p in greedy)
g_sal  = sum(p["sal"]  for p in greedy)
print(f"Greedy lineup: {len(greedy)} players  |  ${g_sal:,} salary  |  {g_pts:.1f} pts")


## Step 6 — Compare lineups


In [ ]:
SLOT_ORDER = ["QB", "RB", "WR", "TE", "FLEX", "DST"]

def print_lineup(lp, label):
    print(f"\n{'='*60}")
    print(f"  {label}")
    print(f"{'='*60}")
    print(f"  {'Slot':<6}  {'Name':<27} {'Pos':<5} {'Salary':>8}  {'Pts':>6}")
    print(f"  {'-'*54}")
    smap = {s: i for i, s in enumerate(SLOT_ORDER)}
    for p in sorted(lp, key=lambda x: smap.get(x["slot"], 99)):
        print(f"  {p['slot']:<6}  {p['name']:<27} {p['pos']:<5} ${p['sal']:>7,}  {p['proj']:>6.1f}")
    print(f"  {'-'*54}")
    tot_sal = sum(p['sal']  for p in lp)
    tot_pts = sum(p['proj'] for p in lp)
    print(f"  {'TOTAL':<6}  {'':27} {'':5} ${tot_sal:>7,}  {tot_pts:>6.1f}")
    print(f"  Remaining cap: ${SALARY_CAP - tot_sal:,}")
    print(f"{'='*60}")
    return tot_pts

q_pts = print_lineup(lineup, "QUANTUM LINEUP  (Qumulator KLT solver)")
g_pts = print_lineup(greedy, "CLASSICAL LINEUP  (Greedy pts/$ heuristic)")

diff  = q_pts - g_pts
sign  = "+" if diff >= 0 else ""
print(f"\nQuantum advantage: {sign}{diff:.1f} projected points")
if diff > 0:
    print(f"Quantum lineup is BETTER by {diff:.1f} pts")
elif diff < 0:
    print(f"Greedy wins by {-diff:.1f} pts")
else:
    print("Tied")


## Step 7 — Visualise


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

POS_COLORS = {"QB": "#ff6b6b", "RB": "#ffd93d", "WR": "#6bcb77", "TE": "#4d96ff", "DST": "#c77dff"}

fig, axes = plt.subplots(1, 2, figsize=(16, 7), facecolor="#0f0f23")

def draw_lineup_chart(ax, lp, title):
    smap = {s: i for i, s in enumerate(SLOT_ORDER)}
    lp_s = sorted(lp, key=lambda x: smap.get(x["slot"], 99))
    names  = [f"{p['name']}\n({p['slot']})" for p in lp_s]
    pts    = [p["proj"] for p in lp_s]
    colors = [POS_COLORS.get(p["pos"], "#aaaaaa") for p in lp_s]
    y      = list(range(len(names)))

    bars = ax.barh(y, pts, color=colors, alpha=0.85, edgecolor="white", height=0.72)
    for bar, pt in zip(bars, pts):
        ax.text(bar.get_width() + 0.15, bar.get_y() + bar.get_height() / 2,
                f"{pt:.1f}", va="center", color="white", fontsize=10, fontweight="bold")

    ax.set_yticks(y); ax.set_yticklabels(names, color="white", fontsize=9)
    ax.set_xlabel("Projected Points", color="white")
    tot_pts = sum(pts); tot_sal = sum(p["sal"] for p in lp_s)
    ax.set_title(f"{title}\nTotal: {tot_pts:.1f} pts  |  Salary: ${tot_sal:,}", color="white", fontsize=11)
    ax.set_facecolor("#1a1a2e"); ax.tick_params(colors="white")
    ax.set_xlim(0, max(pts) * 1.25)
    for spine in ax.spines.values(): spine.set_edgecolor("#444466")
    ax.invert_yaxis()

draw_lineup_chart(axes[0], lineup, "Quantum Lineup  (Qumulator KLT)")
draw_lineup_chart(axes[1], greedy, "Classical Lineup  (Greedy)")

legend_patches = [mpatches.Patch(color=c, label=pos) for pos, c in POS_COLORS.items()]
fig.legend(handles=legend_patches, loc="lower center", ncol=5, fontsize=9,
           facecolor="#1a1a2e", edgecolor="#444466", labelcolor="white")

diff = sum(p["proj"] for p in lineup) - sum(p["proj"] for p in greedy)
sign = "+" if diff >= 0 else ""
plt.suptitle(
    f"Fantasy Football Lineup Optimizer\n"
    f"Quantum: {sum(p['proj'] for p in lineup):.1f} pts  vs  "
    f"Classical: {sum(p['proj'] for p in greedy):.1f} pts  "
    f"({sign}{diff:.1f} quantum advantage)",
    color="white", fontsize=13)
plt.tight_layout(rect=[0, 0.06, 1, 0.93])
plt.savefig("fantasy_football_output.png", dpi=120, bbox_inches="tight", facecolor="#0f0f23")
plt.show()
print("Plot saved: fantasy_football_output.png")


In [ ]:
# Bonus: KLT confidence scores for all players --------------------------------
klt_conf = np.cos(phases)   # +1 = strongly selected, -1 = strongly excluded
order    = np.argsort(-klt_conf)
selected_set = {p["idx"] for p in lineup}

fig, ax = plt.subplots(figsize=(14, 10), facecolor="#0f0f23")
names_s  = [PLAYER_POOL[i][1] for i in order]
pos_s    = [PLAYER_POOL[i][0] for i in order]
conf_s   = [klt_conf[i]       for i in order]
colors_s = [POS_COLORS.get(p, "#aaaaaa") for p in pos_s]
alphas   = [0.95 if order[i] in selected_set else 0.38 for i in range(len(order))]

for i, (n_bar, c, col, a) in enumerate(zip(names_s, conf_s, colors_s, alphas)):
    ax.barh(i, c, color=col, alpha=a, edgecolor="white", linewidth=0.3, height=0.78)

ax.axvline(0, color="white", linewidth=1, alpha=0.5)
ax.set_yticks(range(len(names_s)))
ax.set_yticklabels([f"{n}  ({p})" for n, p in zip(names_s, pos_s)], color="white", fontsize=8)
ax.set_xlabel("KLT Selection Confidence  cos(phi_i)", color="white")
ax.set_title("KLT Ground-State Confidence Scores\n(bright = selected in final lineup)", color="white", fontsize=11)
ax.set_facecolor("#1a1a2e"); ax.tick_params(colors="white")
for spine in ax.spines.values(): spine.set_edgecolor("#444466")
ax.invert_yaxis()
legend_patches = [mpatches.Patch(color=c, label=pos, alpha=0.9) for pos, c in POS_COLORS.items()]
ax.legend(handles=legend_patches, fontsize=9, facecolor="#1a1a2e", edgecolor="#444466", labelcolor="white")

fig.patch.set_facecolor("#0f0f23")
plt.tight_layout()
plt.savefig("fantasy_klt_confidence.png", dpi=120, bbox_inches="tight", facecolor="#0f0f23")
plt.show()
print("Plot saved: fantasy_klt_confidence.png")


## Summary

The KLT quantum solver explores the full combinatorial space of lineup combinations
simultaneously. Unlike greedy, it considers the **joint effect** of all salary and
position constraints at once, finding lineups that position-by-position heuristics miss.

**Tuning the result:**
- Increase `LAMBDA_POS` or `LAMBDA_CAP` if the lineup violates constraints
- Decrease them if the lineup is too conservative (too much salary cap left over)
- Run multiple times with slightly different `confinement_strength` for lineup diversity

**Powered by [Qumulator](https://qumulator.com)** - quantum optimisation on classical hardware.
